# Description

In this notebook, we test generation of empirical bench data.

In [ ]:
from __future__ import annotations

import csv
import time
from pathlib import Path

import numpy as np
import sympy as sp
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from config.empiricalbench_config import EBENCH, CEQL_TRAIN, CEQL
from empirical_bench_data.empirical_bench import load, gen_dataset

from src.ComplexEQL import ComplexEQL
from src.utils import set_seed, train


def train_test_split(X: np.ndarray, y: np.ndarray, test_size: float, seed: int):
    n = X.shape[0]
    rs = np.random.RandomState(seed)
    idx = np.arange(n)
    rs.shuffle(idx)
    n_te = int(round(test_size * n))
    te = idx[:n_te]
    tr = idx[n_te:]
    return X[tr], y[tr], X[te], y[te]


def main():
    out_csv = Path(EBENCH.results_csv_path)
    out_csv.parent.mkdir(parents=True, exist_ok=True)

    data = load()

    keys = list(EBENCH.keys)
    if len(keys) == 0:
        keys = [p["key"] for p in data["problems"].values()]

    with out_csv.open("w", newline="") as f:
        w = csv.writer(f)
        w.writerow(
            [
                "key",
                "run",
                "seed",
                "n_samples",
                "n_features",
                "train_mse",
                "test_mse",
                "duration_s",
                "expr",
            ]
        )

        for key in keys:
            if key in ["hubble"]:
                continue
            X_df, y = gen_dataset(data, key, scale_dataset=True)
            X_np = np.asarray(X_df.values, dtype=np.float32)
            y_np = np.asarray(y, dtype=np.float32).reshape(-1)

            for run_i in range(int(EBENCH.n_runs)):
                seed = int(EBENCH.base_seed) + run_i
                set_seed(seed)

                Xtr, ytr, Xte, yte = train_test_split(
                    X_np, y_np, test_size=float(EBENCH.test_size), seed=int(EBENCH.split_seed) + seed
                )

                CEQL.n_input_fields = int(Xtr.shape[1])
                device = torch.device(getattr(CEQL_TRAIN, "device", "cpu"))

                Xtr_t = torch.tensor(Xtr, device=device)
                ytr_t = torch.tensor(ytr.reshape(-1, 1), device=device)

                dl = DataLoader(
                    TensorDataset(Xtr_t, ytr_t),
                    batch_size=int(getattr(CEQL_TRAIN, "train_batch_size", 2**14)),
                    shuffle=True,
                    drop_last=False,
                )

                model = ComplexEQL(CEQL).to(device)
                loss_fn = nn.MSELoss()
                opt = torch.optim.Adam(model.parameters(), lr=float(getattr(CEQL_TRAIN, "lr", 1e-3)))

                sched = None
                if getattr(CEQL_TRAIN, "scheduler", None) == "ReduceLROnPlateau":
                    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
                        opt, **getattr(CEQL_TRAIN, "schedulerparams", {})
                    )

                t0 = time.perf_counter()
                model, _ = train(
                    model=model,
                    dataloader=dl,
                    optimizer=opt,
                    loss_fn=loss_fn,
                    cfg=CEQL_TRAIN,
                    device=device,
                    scheduler=sched,
                )
                dur = time.perf_counter() - t0

                model.eval()
                with torch.no_grad():
                    yhat_tr = model(torch.tensor(Xtr, device=device)).real.squeeze(-1).cpu().numpy()
                    yhat_te = model(torch.tensor(Xte, device=device)).real.squeeze(-1).cpu().numpy()

                train_mse = float(np.mean((yhat_tr - ytr) ** 2))
                test_mse = float(np.mean((yhat_te - yte) ** 2))

                expr = None
                try:
                    syms = [sp.Symbol(str(c)) for c in X_df.columns[: CEQL.n_input_fields]]
                    expr = model.get_symbolic_expression(syms, rounding_decimals=5, use_imag=False)
                except Exception:
                    expr = None

                expr_str = "" if expr is None else str(expr)

                w.writerow(
                    [
                        key,
                        run_i,
                        seed,
                        int(X_np.shape[0]),
                        int(X_np.shape[1]),
                        train_mse,
                        test_mse,
                        dur,
                        expr_str,
                    ]
                )
                f.flush()

                print(
                    f"[{key}] run={run_i} seed={seed} "
                    f"train_mse={train_mse:.3e} test_mse={test_mse:.3e} dur={dur:.2f}s"
                )
                print("Found expression:", expr_str)


if __name__ == "__main__":
    main()
